# Project imports

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

from datetime import datetime
import time
from fontTools.merge.util import current_time

# initialize flappy game

In [2]:
import sys
import os
os.environ['SDL_VIDEODRIVER'] = 'dummy'
sys.path.insert(0, '../itml-project2')
# noinspection PyUnresolvedReferences
from ple.games.flappybird import FlappyBird
# noinspection PyUnresolvedReferences
from ple import PLE

pygame 2.6.1 (SDL 2.28.4, Python 3.13.0)
Hello from the pygame community. https://www.pygame.org/contribute.html
couldn't import doomish


In [3]:

class AdaptiveLayer(nn.Module):
    """Per-neuron learnable combination of basis functions.

    Each neuron computes: sum_i( w_i * basis_i(z) )

    Bases: z, z², cos(z), sin(z), talpha(z)*z
    """
    BASIS_NAMES = ['z', 'z²', 'cos(z)', 'sin(z)', 'talpha(z)*z']
    NUM_BASES = 5

    def __init__(self, features, a=1000):
        super().__init__()
        self.features = features

        # TAlpha params (for the talpha*z basis)
        self.a = a
        self.c = nn.Parameter(torch.full((features,), 0.5))
        self.d = nn.Parameter(torch.zeros(features))
        self.b = nn.Parameter(torch.ones(features))

        # Mixing weights per neuron — init biased toward linear (identity)
        w_init = torch.zeros(features, self.NUM_BASES)
        w_init[:, 0] = 1.0
        self.weights = nn.Parameter(w_init)

    def _talpha(self, z):
        u = self.a * z / 2 - self.d * self.a
        return self.c * (torch.tanh(u / 2) - 1) + self.b

    def forward(self, z):
        bases = torch.stack([
            z,                        # linear
            z ** 2,                   # quadratic
            torch.cos(z),            # cosine
            torch.sin(z),            # sine
            self._talpha(z) * z,     # learnable step * input
        ], dim=-1)                    # (batch, features, 5)

        return (bases * self.weights).sum(dim=-1)  # (batch, features)


class PPO_Flappy(nn.Module):

    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(8, 64)
        self.adaptive1 = AdaptiveLayer(64)
        self.fc2 = nn.Linear(64, 64)
        self.adaptive2 = AdaptiveLayer(64)
        self.fc3 = nn.Linear(64, 64)
        self.adaptive3 = AdaptiveLayer(64)

        self.actor = nn.Linear(64, 2)
        self.critic = nn.Linear(64, 1)

    def forward(self, z):
        z = self.adaptive1(self.fc1(z))
        z = self.adaptive2(self.fc2(z))
        z = self.adaptive3(self.fc3(z))

        action_prob = F.softmax(self.actor(z), dim=-1)
        state_value = self.critic(z)
        return action_prob, state_value


# Hyper parameters

In [4]:
lr = 0.0005
epochs = 200
K_epochs = 3

epsilon = 0.2           # Clipping range
gamma = 0.99            # Reward discount factor
lam = 0.95
target_steps = 8192    # Collect this many steps before training
minibatch_size = int(512**(1/2))   # SGD minibatch size
max_grad_norm = 1.0     # Gradient clipping

# Loss Coeffs
c0 = 1.0                # clip_coef
c1 = 0.05                # value_coef
c2 = 0.001               # entropy_coef


# Model initialization

In [5]:
model = PPO_Flappy()

"""
try:
    model.load_state_dict(torch.load('../flappy weights/TalphaModel.pt'))
except FileNotFoundError:
    print("No weights found in: flappy weights/")
"""

optimizer = torch.optim.Adam(model.parameters(),lr = lr)

# Helper Functions

In [6]:
def normalize_game_state(state):
    means = torch.tensor([150.0, 0.0, 76.0, 108.0, 208.0, 226.0, 108.0, 208.0])
    stds = torch.tensor([44.0, 5.0, 44.0, 48.0, 48.0, 44.0, 48.0, 48.0])
    return (state - means) / stds


def compute_advantage(rewards, values, gamma, lam):
    t_max = len(rewards)
    advantages = torch.zeros(t_max, dtype=torch.float32)
    last_gae = 0.0
    for t in reversed(range(t_max)):
        next_value = values[t + 1] if t < t_max - 1 else 0.0
        delta = rewards[t] + gamma * next_value - values[t]
        last_gae = delta + gamma * lam * last_gae
        advantages[t] = last_gae
    values_targ = advantages + values
    return advantages.detach(), values_targ.detach()


# training Loop

In [ ]:
import time

print_freq = 10

start_time = time.time()
game = FlappyBird()
p = PLE(game, fps=30, display_screen=False, force_fps=True,
        reward_values={'positive': 1.0, 'tick': 0.0, 'loss': -5.0})
p.init()

epoch_pipes = []
all_L_clip = []
all_L_vf = []
all_L_entropy = []

for epoch in range(epochs):

    # Anneal learning rate
    cur_lr = lr * (1.0 - epoch / epochs)
    for param_group in optimizer.param_groups:
        param_group['lr'] = cur_lr

    # --- Collect batch of episodes until target_steps reached ---
    batch_states = []
    batch_actions = []
    batch_logprobs = []
    batch_values = []
    batch_rewards = []
    batch_ep_lengths = []
    total_steps = 0
    episode_pipes = []

    while total_steps < target_steps:
        p.reset_game()
        ep_states, ep_actions, ep_logprobs, ep_values, ep_rewards = [], [], [], [], []
        nr_pipes = 0

        while not p.game_over():
            state = normalize_game_state(torch.tensor(list(p.getGameState().values())))
            with torch.no_grad():
                action_prob, critic_val = model(state)

            dist = torch.distributions.Categorical(action_prob.squeeze())
            action = dist.sample()

            reward = p.act(p.getActionSet()[action.item()])
            if reward > 0:
                nr_pipes += 1

            ep_states.append(state)
            ep_actions.append(action)
            ep_logprobs.append(dist.log_prob(action))
            ep_values.append(critic_val.squeeze())
            ep_rewards.append(reward)

        episode_pipes.append(nr_pipes)
        ep_len = len(ep_states)
        total_steps += ep_len

        batch_states.append(torch.stack(ep_states))
        batch_actions.append(torch.stack(ep_actions))
        batch_logprobs.append(torch.stack(ep_logprobs))
        batch_values.append(torch.stack(ep_values))
        batch_rewards.append(torch.tensor(ep_rewards, dtype=torch.float32))
        batch_ep_lengths.append(ep_len)

    avg_pipes = sum(episode_pipes) / len(episode_pipes)
    epoch_pipes.append(avg_pipes)

    # --- Compute GAE advantages per episode, then concatenate ---
    all_adv = []
    all_vtarg = []
    for i in range(len(batch_rewards)):
        adv, vtarg = compute_advantage(batch_rewards[i], batch_values[i], gamma, lam)
        all_adv.append(adv)
        all_vtarg.append(vtarg)

    states = torch.cat(batch_states)
    actions = torch.cat(batch_actions)
    logp_old = torch.cat(batch_logprobs)
    advantages = torch.cat(all_adv)
    values_targ = torch.cat(all_vtarg)

    # Normalize advantages
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

    # --- Minibatch PPO training ---
    for k in range(K_epochs):
        indices = torch.randperm(states.shape[0])
        for start in range(0, states.shape[0] - minibatch_size + 1, minibatch_size):
            mb = indices[start:start + minibatch_size]

            action_probs, new_values = model(states[mb])
            new_dist = torch.distributions.Categorical(action_probs)
            new_logp = new_dist.log_prob(actions[mb])

            ratio = torch.exp(new_logp - logp_old[mb])
            surr1 = ratio * advantages[mb]
            surr2 = torch.clamp(ratio, 1 - epsilon, 1 + epsilon) * advantages[mb]

            L_clip = torch.min(surr1, surr2).mean()
            L_vf = F.mse_loss(new_values.squeeze(), values_targ[mb])
            entropy = new_dist.entropy().mean()

            loss = -(c0 * L_clip - c1 * L_vf + c2 * entropy)

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()

        all_L_clip.append(L_clip.item())
        all_L_vf.append(L_vf.item())
        all_L_entropy.append(entropy.item())

    # --- Print stats ---
    if (epoch + 1) % print_freq == 0:
        elapsed = time.time() - start_time
        n = min(print_freq, len(epoch_pipes))
        recent_pipes = epoch_pipes[-n:]
        n_loss = min(K_epochs * print_freq, len(all_L_clip))

        print(f"Epoch {epoch+1:5d} | "
              f"Avg Pipes: {sum(recent_pipes)/len(recent_pipes):7.2f} | "
              f"L_clip: {sum(all_L_clip[-n_loss:])/len(all_L_clip[-n_loss:]):.4f} | "
              f"L_vf: {sum(all_L_vf[-n_loss:])/len(all_L_vf[-n_loss:]):.4f} | "
              f"L_ent: {sum(all_L_entropy[-n_loss:])/len(all_L_entropy[-n_loss:]):.4f} | "
              f"lr: {cur_lr:.6f} | "
              f"Time: {elapsed:.1f}s")


libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile


Epoch    10 | Avg Pipes:    3.25 | L_clip: -0.0054 | L_vf: 0.2216 | L_ent: 0.5351 | lr: 0.000498 | Time: 28.7s
Epoch    20 | Avg Pipes:  136.38 | L_clip: -0.0653 | L_vf: 0.1619 | L_ent: 0.4092 | lr: 0.000495 | Time: 74.9s
Epoch    30 | Avg Pipes:   48.62 | L_clip: -0.0111 | L_vf: 0.3824 | L_ent: 0.3976 | lr: 0.000493 | Time: 106.7s
Epoch    40 | Avg Pipes:   93.47 | L_clip: 0.0190 | L_vf: 0.3824 | L_ent: 0.3680 | lr: 0.000490 | Time: 143.9s
Epoch    50 | Avg Pipes:   27.29 | L_clip: 0.0349 | L_vf: 0.2306 | L_ent: 0.4065 | lr: 0.000488 | Time: 176.7s
Epoch    60 | Avg Pipes:   59.63 | L_clip: -0.0536 | L_vf: 0.3100 | L_ent: 0.4047 | lr: 0.000485 | Time: 208.8s
Epoch    70 | Avg Pipes:  156.90 | L_clip: -0.0162 | L_vf: 0.2924 | L_ent: 0.3787 | lr: 0.000483 | Time: 255.6s


# Plotting Stats

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 5))
ax.scatter(range(len(epoch_pipes)), epoch_pipes, s=1, alpha=0.3, label='Avg pipes per epoch')

# Rolling average
window = 50
if len(epoch_pipes) >= window:
    rolling_avg = [sum(epoch_pipes[max(0,i-window):i+1]) / len(epoch_pipes[max(0,i-window):i+1])
                   for i in range(len(epoch_pipes))]
    ax.plot(rolling_avg, color='red', linewidth=1.5, label=f'Rolling avg (window={window})')

ax.set_xlabel('Epoch')
ax.set_ylabel('Avg Pipes')
ax.set_title(f'Training Progress — Best Avg: {max(epoch_pipes):.1f} pipes')
ax.legend()
plt.tight_layout()
plt.show()


# Viewing model playing Flappy

In [ ]:
# Number of games to watch
import pygame
num_runs = 5

# Enable actual display (training set SDL_VIDEODRIVER to dummy)
os.environ.pop('SDL_VIDEODRIVER', None)
pygame.quit()
pygame.init()

model.eval()
game_vis = FlappyBird()
p_vis = PLE(game_vis, fps=30, display_screen=True, force_fps=False)
p_vis.init()

for run in range(num_runs):
    p_vis.reset_game()
    total_reward = 0
    while not p_vis.game_over():
        state = normalize_game_state(torch.tensor(list(p_vis.getGameState().values())))
        with torch.no_grad():
            action_prob, _ = model(state)
        action = (action_prob == max(action_prob))

        if action[0] == True:
            ret_action = p_vis.getActionSet()[0]
        else:
            ret_action = p_vis.getActionSet()[1]

        reward = p_vis.act(ret_action)
        total_reward += reward
    print(f"Run {run+1}/{num_runs} - Total reward: {total_reward:.1f}")

pygame.quit()
model.train()
print("Done.")

# Saving Model

In [ ]:
torch.save(model.state_dict(), "../flappy weights/TalphaModel.pt")